In [1]:
# !pip install langchain
# !pip install langchain-community
# !pip install pypdf

In [2]:
import pprint

In [3]:
# Import library
from langchain_community.document_loaders import PyPDFLoader, UnstructuredHTMLLoader

# Create a document loader for rag_paper.pdf
loader = PyPDFLoader('pdf_data.pdf')

# Load the document
pdf_data = loader.load()

print(pdf_data[0].page_content)
print(pdf_data[0].metadata)

/tmp/ipykernel_1643/3845071257.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, UnstructuredHTMLLoader
/home/codespace/.python/current/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèmes de management de la sécurité de l'information — 
Exigences
INTERNATIONAL 
STANDARD
ISO/IEC 
27001
Third edition  
2022-10
Reference number 
ISO/IEC 27001:2022(E)
© ISO/IEC 2022
--``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`---
{'producer': 'PDPreStamp v3.3', 'creator': 'Adobe InDesign 16.4 (Windows)', 'creationdate': '2022-09-29T17:54:00+02:00', 'author': 'ISO', 'license': 'Information Handling Services, 2022', 'moddate': '2022-10-26T19:22:40+08:00', 'title': 'ISO/IEC 27001:2022', 'trapped': '/False', 'source': 'pdf_data.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}


In [4]:
# !pip install unstructured

In [5]:
# Import library
from langchain_community.document_loaders import UnstructuredHTMLLoader

# Load the document
html_data = UnstructuredHTMLLoader('html_data.html').load()

# Print the first document's content
print(html_data[0].page_content)

# Print the first document's metadata
print(html_data[0].metadata)

Retrieval-Augmented Generation (RAG)

By Shikher AI Blog

What is RAG?

Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation. It allows language models to access external knowledge sources.

Why RAG is Important?

Traditional LLMs rely only on training data. RAG improves accuracy by retrieving relevant documents during inference.

How it Works

Step 1: Load documents

Step 2: Split into chunks

Step 3: Generate embeddings

Step 4: Store in vector database

Step 5: Retrieve and generate answers

© 2026 AI Learning Platform
{'source': 'html_data.html'}


In [6]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path="csv_data.csv")
csv_data = loader.load()

print(csv_data[0].page_content)
print(csv_data[-1].metadata)

id: 1
title: What is RAG
category: AI
content: Retrieval-Augmented Generation combines retrieval with generation to improve LLM accuracy.
{'source': 'csv_data.csv', 'row': 4}


In [7]:
# !pip install langchain_text_splitters

In [8]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator='.',
    chunk_size=75,  
    chunk_overlap=10  
)
# Split the text using text_splitter
chunks = text_splitter.split_text(csv_data[0].page_content)
pprint.pprint(chunks)
pprint.pprint([len(chunk) for chunk in chunks])

['id: 1\n'
 'title: What is RAG\n'
 'category: AI\n'
 'content: Retrieval-Augmented Generation combines retrieval with generation '
 'to improve LLM accuracy']
[136]


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=['.'],
    chunk_size=75,
    chunk_overlap=10
)

# Split the text using text_splitter
chunks = text_splitter.split_text(csv_data[-1].page_content)
# chunks = text_splitter.split_documents(pdf_data)
pprint.pprint(chunks)
print([len(chunk) for chunk in chunks])

['id: 5\n'
 'title: Chunking Strategy\n'
 'category: AI\n'
 'content: Splitting documents into chunks improves retrieval performance in '
 'RAG systems',
 '.']
[130, 1]


In [10]:
# !pip install langchain_chroma langchain_openai

In [11]:
# !pip install langchain_community
# !pip install sentence-transformers

In [12]:
# from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Create an instance of the OpenAIEmbeddings class

# embedding_model = OpenAIEmbeddings(
    # api_key="<OPENAI_API_TOKEN>",
    #   model='text-embedding-3-small')

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create a Chroma vector store and embed the chunks
vector_store = Chroma.from_documents(
    documents=chunks, 
    embedding=embedding_model
)

/tmp/ipykernel_1643/1908502660.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:01<00:00, 86.24it/s]


AttributeError: 'str' object has no attribute 'page_content'

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2},
    search_type="similarity",
    # search_type="mmr",
) 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# prompt = ChatPromptTemplate.from_messages()
prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer the question based on the context provided.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"
)

In [ ]:
# llm = ChatOpenAI(model="gpt-3.5-turbo",api_key="your-api-key", temperature=0)

from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.2",
    device=0  # GPU (use -1 for CPU)
)

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:
result = chain.invoke({"question": "What is the main topic of the document?"})
print(result)

In [ ]:
prompt = """
Use the only the context provided to answer the following question. If you don't know the answer, reply that you are unsure.
Context: {context}
Question: {question}
"""

from langchain_core.prompts import ChatPromptTemplate


# Convert the string into a chat prompt template
prompt_template = ChatPromptTemplate.from_template(prompt)

# Create an LCEL chain to test the prompt
chain = prompt_template | llm

# Invoke the chain on the inputs provided
print(chain.invoke({"context": "DataCamp's RAG course was created by Meri Nova and James Chapman!", "question": "Who created DataCamp's RAG course?"}))

NameError: name 'llm' is not defined

In [ ]:
# Convert the vector store into a retriever
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":2})

# Create the LCEL retrieval chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

# Invoke the chain
print(chain.invoke("Who are the authors?"))

NameError: name 'RunnablePassthrough' is not defined